# Hyperparameter Tuning
**CubicNova Data Science Roadmap — Module 11**

This notebook demonstrates GridSearchCV, RandomizedSearchCV, pipeline tuning, and comparing cross-validation results with final test performance. Run cells from top to bottom.

## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_iris, make_classification
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score
from scipy.stats import randint

## 2. Establish a baseline

In [ ]:
X, y = load_iris(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

## 3. GridSearchCV with Decision Tree

In [ ]:
param_grid = {
    "max_depth": [2, 3, 5, None],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
}
grid_search = GridSearchCV(
    DecisionTreeClassifier(random_state=42),
    param_grid=param_grid,
    cv=cv,
    scoring="accuracy",
    n_jobs=-1,
    refit=True,
)
grid_search.fit(X_train, y_train)
print("Best parameters:", grid_search.best_params_)
print("Best CV accuracy:", round(grid_search.best_score_, 3))
results = pd.DataFrame(grid_search.cv_results_).sort_values("mean_test_score", ascending=False)
display(results[["params", "mean_test_score", "std_test_score", "rank_test_score"]].head(10))
pred = grid_search.best_estimator_.predict(X_test)
print("Test accuracy:", round(accuracy_score(y_test, pred), 3))
print(confusion_matrix(y_test, pred))

## 4. RandomizedSearchCV with Random Forest

In [ ]:
Xb, yb = make_classification(
    n_samples=1000, n_features=20, n_informative=8, n_redundant=4,
    weights=[0.8, 0.2], random_state=42
)
X_train_b, X_test_b, y_train_b, y_test_b = train_test_split(
    Xb, yb, test_size=0.2, random_state=42, stratify=yb
)
distributions = {
    "n_estimators": randint(100, 301),
    "max_depth": [None, 5, 10, 20],
    "min_samples_split": randint(2, 12),
    "min_samples_leaf": randint(1, 6),
    "max_features": ["sqrt", "log2", None],
}
random_search = RandomizedSearchCV(
    RandomForestClassifier(random_state=42),
    param_distributions=distributions,
    n_iter=12,
    cv=cv,
    scoring="average_precision",
    random_state=42,
    n_jobs=-1,
    refit=True,
)
random_search.fit(X_train_b, y_train_b)
print("Best parameters:", random_search.best_params_)
print("Best CV average precision:", round(random_search.best_score_, 3))
pred_b = random_search.predict(X_test_b)
proba_b = random_search.predict_proba(X_test_b)[:, 1]
print(classification_report(y_test_b, pred_b, zero_division=0))
print("Test ROC-AUC:", round(roc_auc_score(y_test_b, proba_b), 3))

## 5. Tune a pipeline parameter

In [ ]:
pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(max_iter=2000)),
])
pipe_grid = {"classifier__C": [0.001, 0.01, 0.1, 1, 10, 100]}
pipe_search = GridSearchCV(
    pipe, pipe_grid, cv=cv, scoring="accuracy", n_jobs=-1
)
pipe_search.fit(X_train, y_train)
print("Best pipeline parameters:", pipe_search.best_params_)
print("Best CV accuracy:", round(pipe_search.best_score_, 3))
print("Test accuracy:", round(pipe_search.score(X_test, y_test), 3))

## 6. Interpret the results

- `best_params_`: selected configuration.
- `best_score_`: mean CV score for the selected configuration.
- `cv_results_`: results for all searched candidates.
- The test score is a separate final estimate and may differ from the CV score.
- Scikit-learn search scorers follow “higher is better”; loss scorers such as MAE use a `neg_` prefix.
- Use a `Pipeline` so preprocessing is learned within each training fold.
- Tune with training/CV data and reserve the final test set for one final evaluation.

## Practice

Complete `practice.md` and compare with `solution.md`.

## Further reading

- [Scikit-learn hyperparameter tuning](https://scikit-learn.org/stable/modules/grid_search.html)
- [GridSearchCV](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html)
- [RandomizedSearchCV](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.RandomizedSearchCV.html)